# PhishGuard AI - Data Analysis
Explores `dataset/unified_threat_dataset.csv`, checks class balance and feature behaviour, and shows the trained models' metrics.

> If `source_dataset` says **SYNTHETIC**, every number below comes from artificial template data and says nothing about real-world accuracy.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data_cleaning import load_dataset, clean_text
from src.url_features import features_dataframe
sns.set_theme(style="whitegrid")
df = load_dataset(ROOT / "dataset" / "unified_threat_dataset.csv")
print(df.shape); df.head()

## 1. Data sources and class balance

In [ ]:
print(df["source_dataset"].value_counts(), "\n")
pd.crosstab(df["source_type"], df["label"].map({0: "safe", 1: "malicious"}))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
df["label"].map({0: "safe", 1: "malicious"}).value_counts().plot.bar(ax=ax[0], title="Overall balance", rot=0)
pd.crosstab(df["source_type"], df["label"]).plot.bar(ax=ax[1], title="By source type", rot=0)
plt.tight_layout(); plt.show()

## 2. Split check (no leakage between splits)

In [ ]:
print(pd.crosstab(df["split"], df["label"]))
dup = df.assign(k=df["text"].str.lower() + "||" + df["url"].str.lower()).duplicated("k").sum()
print("duplicate rows left:", dup)

## 3. URL features: safe vs malicious

In [ ]:
urls = df[df["url"] != ""].reset_index(drop=True)
X = features_dataframe(urls["url"]); X["label"] = urls["label"]
X.groupby("label").mean().T.round(3)

In [ ]:
cols = ["url_length", "num_hyphens", "num_subdomains", "host_entropy"]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for a, c in zip(axes, cols):
    sns.boxplot(data=X, x="label", y=c, ax=a); a.set_xticks([0, 1]); a.set_xticklabels(["safe", "malicious"]); a.set_title(c)
plt.tight_layout(); plt.show()

In [ ]:
flags = ["has_ip_host", "has_at_symbol", "brand_mismatch", "suspicious_tld", "has_shortener", "uses_https"]
X.groupby("label")[flags].mean().T.rename(columns={0: "safe", 1: "malicious"}).plot.barh(figsize=(7, 3.5), title="Share of URLs with each flag")
plt.tight_layout(); plt.show()

## 4. Text: length and most frequent words

In [ ]:
txt = df[df["text"] != ""].copy()
txt["words"] = txt["text"].str.split().str.len()
print(txt.groupby("label")["words"].describe().round(1))
from collections import Counter
for lab, name in ((1, "malicious"), (0, "safe")):
    c = Counter(w for t in txt.loc[txt.label == lab, "text"] for w in clean_text(t).split() if len(w) > 3)
    print(name, "->", [w for w, _ in c.most_common(15)])

## 5. Trained models

In [ ]:
for name in ("url", "text"):
    p = ROOT / "models" / f"{name}_metrics.json"
    if p.exists():
        m = json.loads(p.read_text())
        print(name.upper(), m["model_name"], "| synthetic:", m["is_synthetic"])
        print({k: v for k, v in m["metrics"]["test"].items() if k != "confusion_matrix"}, "\n")
    else:
        print(name, "model not trained yet: python train_all.py")

In [ ]:
from src.url_detector import URLDetector
imp = pd.Series(URLDetector().bundle["feature_importance"]).head(12)
imp.iloc[::-1].plot.barh(figsize=(7, 4), title="URL model - top feature importances"); plt.tight_layout(); plt.show()